# Chương 1 — Milestone 1: Data Crawling

Báo cáo tổng hợp từ **dữ liệu và bằng chứng đã xác minh**. Bảng và biểu đồ được xây dựng tại thời điểm Jupyter Book biên dịch, không sử dụng số liệu giả hoặc ảnh chụp.

## 1.1. Phương pháp thu thập

**Entry discovery:** truy vấn arXiv API theo khoảng ID đã phân công, lưu metadata cấp publication và số version. **Full-text:** tải từng `/src/{arxiv_id}vN` qua `export.arxiv.org`, giữ nguyên nội dung `.tex/.bib`, tạo marker và hash để kiểm chứng. **References:** lưu cả raw reference entries của Semantic Scholar (kể cả tài liệu không có arXiv ID), đồng thời duy trì bảng cạnh arXiv-only phục vụ các milestone sau.

References từ Semantic Scholar có phạm vi **publication**, không thể tự gán cho version lịch sử. Một số ngày revision trung gian vẫn chưa có nguồn xác minh. Phiên bản PDF-only được giữ trạng thái riêng, không tính là TeX thành công.

## 1.2. Kết quả và phân tích corpus

Các chỉ số dưới đây được nạp từ `m1_summary.json` sau khi chạy cell 2.7 và 2.9 trên Colab, xuất tại `data/reports/`. Bản tóm tắt này phải được chuyển đến môi trường build của **repository Jupyter Book riêng** (có thể thiết lập biến môi trường `M1_SUMMARY_JSON`). Nếu không cung cấp dữ liệu, chương này sẽ hiển thị trạng thái thiếu số liệu thay vì số liệu minh họa.


In [ ]:
import os
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

custom = os.environ.get("M1_SUMMARY_JSON", "").strip()
summary_path = Path(custom).expanduser() if custom else Path("_data/m1_summary.json")
if not summary_path.is_file():
    display(Markdown(
        "**Chưa có số liệu M1 được xác minh trong môi trường build.** "
        "Chạy Colab cell 2.7 → 2.9, xuất m1_summary.json, "
        "rồi cung cấp đường dẫn qua M1_SUMMARY_JSON. "
        "Không sử dụng biểu đồ demo thay dữ liệu thực."
    ))
else:
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    assert summary.get("schema_version") == 1, "Unexpected summary schema"
    coverage = summary["coverage"]
    sources = summary["source_statuses"]
    display(Markdown(
        f"**Thời điểm tổng hợp:** {summary.get('generated_at','unknown')} "
        f"| **Tình trạng:** {summary.get('integrity_status','unknown')} "
        f"| **Nguồn:** {summary_path}"
    ))
    frame = pd.DataFrame([
        {"Chỉ số": "Paper có metadata hợp lệ",
         "Giá trị": coverage["valid_metadata_papers"]},
        {"Chỉ số": "Version dự kiến", "Giá trị": coverage["expected_versions"]},
        {"Chỉ số": "Version source đã xác minh",
         "Giá trị": coverage["verified_source_versions"]},
        {"Chỉ số": "Tỷ lệ source version đã xác minh (%)",
         "Giá trị": coverage["source_coverage_pct"]},
        {"Chỉ số": "Raw reference papers được xác minh",
         "Giá trị": coverage["raw_reference_papers_verified"]},
    ])
    display(frame)
    labels = {
        "verified_success": "TeX verified",
        "verified_pdf_only": "PDF only",
        "pending": "Pending",
        "failed": "Failed",
        "not_found": "HTTP 404",
        "conflicts": "Conflict",
        "missing_files": "Missing files",
    }
    series = pd.Series({v: sources[k] for k, v in labels.items()})
    ax = series.plot.bar(figsize=(9, 4), ylabel="Số phiên bản",
                         title="Trạng thái source theo phiên bản (đã đối chiếu)")
    ax.tick_params(axis="x", labelrotation=35)
    plt.tight_layout()
    plt.show()
    if summary.get("source_file_statistics") is not None:
        display(pd.DataFrame([summary["source_file_statistics"]]).T.rename(
            columns={0: "Giá trị"}))
    else:
        display(Markdown(
            "**Chưa có thống kê kích thước source đáng tin cậy.** "
            "Xuất lại m1_download_versions.csv bằng cell 2.3, "
            "chạy lại 2.7 và 2.9."
        ))


## 1.3. Benchmark và giới hạn diễn giải

Benchmark theo đặc tả cần được đo trên **Google Colab CPU-only**, gồm thời gian toàn pipeline, trung bình thời gian/paper, thời gian entry discovery, peak/average RAM, peak disk và dung lượng đầu ra. Những giá trị này **không thể suy ra** từ bảng download/references hoặc CSV benchmark legacy của một pipeline khác.

Trong phiên bản hiện tại, cell báo cáo chỉ ghi nhận tình trạng bằng chứng. Chỉ sau khi nhóm thực hiện và lưu một lượt chạy đo CPU-only thật sự (với phạm vi/commit/môi trường rõ ràng) mới điền benchmark chính thức. Không sử dụng các con số giả hay bảng minh họa.


In [ ]:
if not summary_path.is_file():
    display(Markdown("**Benchmark:** chưa có summary; chưa được phép kết luận."))
else:
    benchmark = summary["benchmark_evidence"]
    display(Markdown(
        f"**Benchmark của pipeline version-aware:** "
        f"{'chưa được đo/xác minh' if benchmark['official_version_pipeline_metrics'] is None else 'đã có bằng chứng'}.\n\n"
        f"Số dòng benchmark legacy tìm thấy: {benchmark['legacy_run_count']}. "
        "Chúng không được xem là benchmark chính thức của pipeline mới."
    ))
    for requirement in summary["missing_evidence"]:
        display(Markdown(f"- Chưa đủ bằng chứng: {requirement}"))
